In [1]:

from pathlib import Path
from PIL import Image, ImageOps
import pandas as pd
import numpy as np
import json, time

# ---------- Paths ----------
DATA_DIR = Path("/kaggle/input/datasets/pranavsunilwagh/ubiris-v2/ubiris2_1/CLASSES_400_300_Part1")
PROJECT = Path("/kaggle/working/ubiris_v2_project")
OUT_DIR = PROJECT / "notebook02_preprocessing"
IMAGE_DIR = OUT_DIR / "images_224"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f"Dataset path not found: {DATA_DIR}"

# ---------- Load prior metadata when available ----------
meta_path = PROJECT / "notebook01_dataset_analysis" / "UBIRIS_V2_image_metadata.csv"
if meta_path.exists():
    source_df = pd.read_csv(meta_path)
    files = [DATA_DIR / str(p) for p in source_df["filename"] if (DATA_DIR / str(p)).is_file()] if "filename" in source_df.columns else sorted(DATA_DIR.glob("*.tiff"))
else:
    files = sorted(DATA_DIR.glob("*.tiff"))

files = sorted(set(p for p in files if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}))
assert files, "No TIFF images found."

# ---------- Preprocess ----------
start = time.time()
records, errors = [], []

for i, path in enumerate(files, 1):
    try:
        with Image.open(path) as im:
            original_size = list(im.size)
            gray = ImageOps.grayscale(im)
            processed = gray.resize((224, 224), Image.Resampling.LANCZOS)

        # Keep filenames unique and traceable
        out_name = path.stem + ".png"
        out_path = IMAGE_DIR / out_name
        processed.save(out_path, optimize=True)

        # UBIRIS V2 pattern: C<subject>_S<session>_I<image>.tiff
        parts = path.stem.split("_")
        subject = parts[0][1:] if len(parts) == 3 and parts[0].startswith("C") else None
        session = parts[1][1:] if len(parts) == 3 and parts[1].startswith("S") else None
        image_id = parts[2][1:] if len(parts) == 3 and parts[2].startswith("I") else None

        records.append({
            "filename": path.name,
            "relative_path": path.name,
            "processed_path": str(out_path),
            "subject": subject,
            "session": session,
            "image_id": image_id,
            "original_width": original_size[0],
            "original_height": original_size[1],
            "processed_width": 224,
            "processed_height": 224,
            "processed_mode": "L",
            "valid": True
        })
    except Exception as e:
        errors.append({"filename": path.name, "error": str(e)})

    if i % 500 == 0 or i == len(files):
        print(f"Processed {i}/{len(files)} images")

# ---------- Save outputs ----------
metadata = pd.DataFrame(records)
error_df = pd.DataFrame(errors, columns=["filename", "error"])
metadata_path = OUT_DIR / "UBIRIS_V2_preprocessed_metadata.csv"
errors_path = OUT_DIR / "UBIRIS_V2_preprocessing_errors.csv"
summary_path = OUT_DIR / "UBIRIS_V2_preprocessing_summary.json"

metadata.to_csv(metadata_path, index=False)
error_df.to_csv(errors_path, index=False)

summary = {
    "dataset": "UBIRIS V2",
    "source_images_found": len(files),
    "successfully_preprocessed": len(records),
    "failed_images": len(errors),
    "output_dimensions": [224, 224],
    "output_mode": "grayscale",
    "output_format": "PNG",
    "subject_labels_parsed_from_filename": int(metadata["subject"].notna().sum()) if not metadata.empty else 0,
    "processing_seconds": round(time.time() - start, 2),
    "note": "Filename-parsed subject labels are metadata only; identity correctness has not been independently verified."
}
summary_path.write_text(json.dumps(summary, indent=2))

print("\nPREPROCESSING SUMMARY")
print(json.dumps(summary, indent=2))
print("\nSAVED OUTPUTS")
for p in [metadata_path, errors_path, summary_path]:
    print(f"{'OK' if p.exists() else 'MISSING'} | {p.name} | {p.stat().st_size if p.exists() else 0:,} bytes")
print(f"Processed images directory: {IMAGE_DIR}")


Processed 500/5799 images
Processed 1000/5799 images
Processed 1500/5799 images
Processed 2000/5799 images
Processed 2500/5799 images
Processed 3000/5799 images
Processed 3500/5799 images
Processed 4000/5799 images
Processed 4500/5799 images
Processed 5000/5799 images
Processed 5500/5799 images
Processed 5799/5799 images

PREPROCESSING SUMMARY
{
  "dataset": "UBIRIS V2",
  "source_images_found": 5799,
  "successfully_preprocessed": 5799,
  "failed_images": 0,
  "output_dimensions": [
    224,
    224
  ],
  "output_mode": "grayscale",
  "output_format": "PNG",
  "subject_labels_parsed_from_filename": 5799,
  "processing_seconds": 89.15,
  "note": "Filename-parsed subject labels are metadata only; identity correctness has not been independently verified."
}

SAVED OUTPUTS
OK | UBIRIS_V2_preprocessed_metadata.csv | 858,104 bytes
OK | UBIRIS_V2_preprocessing_errors.csv | 15 bytes
OK | UBIRIS_V2_preprocessing_summary.json | 420 bytes
Processed images directory: /kaggle/working/ubiris_v2_pr

# Final Conclusion — UBIRIS V2 Image Preprocessing

## Overview
A total of **5,799 UBIRIS V2 images** were successfully preprocessed into grayscale PNG images of size **224 × 224 pixels**.

## Key Results
- Source images processed: 5,799
- Successfully preprocessed: 5,799
- Failed images: 0
- Output format: PNG
- Output dimensions: 224 × 224
- Output color mode: Grayscale
- Processing time: 89.15 seconds

## Generated Artifacts
- `UBIRIS_V2_preprocessed_metadata.csv`
- `UBIRIS_V2_preprocessing_errors.csv`
- `UBIRIS_V2_preprocessing_summary.json`
- `images_224/` — preprocessed images

## Limitations
Subject, session, and image identifiers were parsed from filenames. These identifiers have not been independently verified as biometric identity labels. Preprocessing standardizes image dimensions and color mode but does not guarantee improved recognition accuracy or image quality.

## Conclusion
The preprocessing stage completed successfully, with all 5,799 readable images converted to the required format and no recorded failures. The generated metadata and summary files provide traceability for subsequent image-quality assessment and recognition experiments.

In [2]:

from pathlib import Path
import json, shutil
from IPython.display import FileLink, display

project = Path("/kaggle/working/ubiris_v2_project")
out = project / "notebook02_preprocessing"
images = out / "images_224"

required = [
    out / "UBIRIS_V2_preprocessed_metadata.csv",
    out / "UBIRIS_V2_preprocessing_errors.csv",
    out / "UBIRIS_V2_preprocessing_summary.json",
]

for p in required:
    assert p.is_file(), f"Missing: {p.name}"
    print(f"OK | {p.name} | {p.stat().st_size:,} bytes")

with open(required[2]) as f:
    summary = json.load(f)

assert summary["successfully_preprocessed"] == 5799
assert summary["failed_images"] == 0
assert len(list(images.glob("*.png"))) == 5799

print("\nVerification passed: 5,799 images, zero failures.")

# Package Notebook 02 outputs
zip_base = "/kaggle/working/UBIRIS_V2_Notebook02_Results"
shutil.make_archive(
    zip_base,
    "zip",
    root_dir=project,
    base_dir="notebook02_preprocessing"
)

zip_path = Path(zip_base + ".zip")
print(f"\nZIP ready: {zip_path.name} ({zip_path.stat().st_size / (1024**2):.1f} MB)")
display(FileLink(str(zip_path), result_html_prefix="Download Notebook 02 results: "))


OK | UBIRIS_V2_preprocessed_metadata.csv | 858,104 bytes
OK | UBIRIS_V2_preprocessing_errors.csv | 15 bytes
OK | UBIRIS_V2_preprocessing_summary.json | 420 bytes

Verification passed: 5,799 images, zero failures.

ZIP ready: UBIRIS_V2_Notebook02_Results.zip (143.2 MB)


/kaggle/working/UBIRIS_V2_Notebook02_Results.zip